# Chapter 5: GPU Architecture and Hardware-Aware Decisions
### Companion notebook (High-Performance Machine Learning)

This notebook makes the chapter's claims measurable on a real GPU. It runs on a
**Colab T4** (Runtime → Change runtime type → T4 GPU) or any recent NVIDIA GPU. The book's numbers
were measured with Python 3.11, PyTorch 2.14.1 (CUDA 13.0), transformers 5.20.2 and sentence-transformers 6.2.0, measured on an NVIDIA A100 80GB (the book's reference), an H100 and a T4.

What you will measure here:

1. **Bandwidth vs compute**: a copy is bound by memory and a large matmul is bound by arithmetic.
2. **Coalescing**: contiguous access is far faster than strided access for the same byte count.
3. **Arithmetic intensity and the roofline**: where the RAG stages fall, computed from real shapes.
4. **Mixed precision (FP16): prefill against decode.** FP16 speeds up the compute-bound prefill matmul and
   does little for memory-bound decode, as Principle #14 predicts.
5. **Roofline plot**: the chapter's roofline with the RAG stages, drawn for the detected GPU.

> The chapter's eight hands-on exercises (Exercises 5.1 to 5.8) are in the book's online resources; the notebook
> exercises in Section 6 are shorter and numbered separately; the companion code is at
> https://github.com/PacktPublishing/High-Performance-Machine-Learning

## 0. Setup

In [ ]:
import torch, time, math, gc
import numpy as np

assert torch.cuda.is_available(), 'Enable a GPU runtime (Colab: Runtime > Change runtime type > T4).'
dev = torch.device('cuda')
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
print(f'GPU: {name}  (compute capability {cap[0]}.{cap[1]})')
print(f'SMs: {props.multi_processor_count}   global mem: {props.total_memory/1e9:.1f} GB')

# Turing (7.5) has FP16 Tensor Cores but no BF16 Tensor Cores; native BF16 starts with Ampere (8.0).
# torch.cuda.is_bf16_supported() also returns True when BF16 runs through emulation, so it is reported separately.
IS_TURING = (cap == (7, 5))
BF16_NATIVE = cap >= (8, 0)
BF16_EMULATED = (not BF16_NATIVE) and torch.cuda.is_bf16_supported()
print(f'Turing-class: {IS_TURING}   native BF16: {BF16_NATIVE}   BF16 by emulation only: {BF16_EMULATED}')
if IS_TURING:
    print('=> On this card, mixed precision means FP16 (Chapter 5, Floating-point formats).')

In [ ]:
def cuda_time(fn, iters=50, warmup=10):
    """Median ms per call, measured with CUDA events."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
        s.record(); fn(); e.record(); torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    return float(np.median(times))

## 1. Bandwidth vs compute

Two kernels. A **copy** does almost no arithmetic per byte, so it should be limited by memory
bandwidth. A large **square matmul** reuses each loaded element across a whole row/column, so it
should approach the compute peak. We measure the effective bandwidth of the first and the achieved
FLOP/s of the second, then compare to the card's ceilings.

In [ ]:
# --- memory-bound: big contiguous copy ---
N = 1 << 26  # 64M floats = 256 MB
x = torch.randn(N, device=dev)
y = torch.empty_like(x)
ms = cuda_time(lambda: y.copy_(x))
bytes_moved = 2 * x.numel() * x.element_size()  # read x + write y
gbs = bytes_moved / (ms * 1e-3) / 1e9
print(f'copy: {ms:.3f} ms   effective bandwidth ~ {gbs:.0f} GB/s')

# --- compute-bound: large FP16 matmul on tensor cores ---
M = 4096
a = torch.randn(M, M, device=dev, dtype=torch.float16)
b = torch.randn(M, M, device=dev, dtype=torch.float16)
ms_mm = cuda_time(lambda: torch.matmul(a, b))
flops = 2 * M**3  # multiply-add per output element
tflops = flops / (ms_mm * 1e-3) / 1e12
print(f'matmul {M}x{M} fp16: {ms_mm:.3f} ms   achieved ~ {tflops:.1f} TFLOP/s')

print('\nInterpretation: the copy sits near the memory ceiling; the matmul sits near the compute',
      'ceiling. Same chip, two different limits (Chapter 5, Performing Roofline analysis).')

# --- self-check: copy is memory-bound, matmul is compute-bound ---
assert gbs > 50, f'effective bandwidth implausibly low: {gbs:.0f} GB/s'
assert tflops > 1, f'matmul throughput implausibly low: {tflops:.1f} TFLOP/s'
print(f'self-check passed: bandwidth ~{gbs:.0f} GB/s, matmul ~{tflops:.1f} TFLOP/s')


## 2. Coalescing: contiguous vs strided

Same number of elements gathered two ways. A contiguous gather lets the hardware merge the loads
into wide transactions; a strided gather scatters them. The byte counts are comparable, so any
timing gap comes from the access pattern (Chapter 5, Coalescing, cache locality, and vectorization; Figure 5.11).

In [ ]:
K = 1 << 22            # 4M elements gathered
base = torch.randn(1 << 26, device=dev)

contig_idx = torch.arange(K, device=dev)
stride = 64
strided_idx = (torch.arange(K, device=dev) * stride) % base.numel()

ms_c = cuda_time(lambda: base.index_select(0, contig_idx))
ms_s = cuda_time(lambda: base.index_select(0, strided_idx))
print(f'contiguous gather: {ms_c:.3f} ms')
print(f'strided gather   : {ms_s:.3f} ms   ({ms_s/ms_c:.1f}x slower for the same element count)')

# --- self-check: strided access is slower than contiguous ---
assert ms_s > ms_c, f'expected strided slower than contiguous, got contig={ms_c:.3f} strided={ms_s:.3f}'
print(f'self-check passed: strided gather is {ms_s/ms_c:.1f}x slower than contiguous')


## 3. Arithmetic intensity and the roofline

Look up the detected card's datasheet peaks (the values in the chapter's Table 5.6), compute its ridge point, then
place the three RAG stages by the arithmetic intensity of their dense linear layers. The shapes are those of the
case study: Gemma-2B (hidden size 2,048, MLP width 16,384, one 256-wide key/value head) for prefill and decode, and
BGE-M3 (hidden size 1,024, MLP width 4,096) for the encoder, whose 512-token chunk becomes 477 token rows.
Substitute your own model's shapes to re-place the points.


In [ ]:
# Datasheet peaks: dense FP16 Tensor Core TFLOP/s and device-memory GB/s (Chapter 5, Table 5.6).
DATASHEET = {
    'T4':   (65.0, 320.0),     # NVIDIA T4
    'A100': (312.0, 2039.0),   # NVIDIA A100 80GB SXM (the A100 40GB has 1,555 GB/s)
    'H100': (989.0, 3350.0),   # NVIDIA H100 SXM (the H100 PCIe has lower peaks)
}
key = next((k for k in DATASHEET if k in name), None)
if key is None:
    key = 'T4'
    print(f'No built-in peaks for {name}: using the T4 values. Enter your GPU\'s datasheet values in DATASHEET.')
PEAK_TFLOPS, PEAK_GBs = DATASHEET[key]
ridge = (PEAK_TFLOPS * 1e12) / (PEAK_GBs * 1e9)
print(f'{key} peaks: {PEAK_TFLOPS:.0f} TFLOP/s FP16 Tensor Core, {PEAK_GBs:.0f} GB/s  -> ridge ~ {ridge:.0f} FLOP/byte')

def linear_ai(M, K, N, bytes_per_el=2):
    """Arithmetic intensity of Y = X W with X (M x K) and W (K x N): 2MKN FLOPs over the bytes of X, W and Y."""
    return 2 * M * K * N / (bytes_per_el * (M * K + K * N + M * N))

# (K, N) of the dense layers
GEMMA = [(2048, 2048), (2048, 256), (2048, 16384), (16384, 2048)]   # Q/O, K/V, MLP up/gate, MLP down
BGE_M3 = [(1024, 1024), (1024, 4096), (4096, 1024)]                  # attention projections, MLP

def stage_range(M, layers):
    vals = [linear_ai(M, K, N) for K, N in layers]
    return min(vals), max(vals)

decode_rng  = stage_range(1, GEMMA)      # batch-one decode: one token row
T = 512
prefill_rng = stage_range(T, [l for l in GEMMA if l[1] != 256])   # 512-token prompt, Q/O and MLP layers
kv_prefill  = linear_ai(T, 2048, 256)    # K/V projections: N = 256 limits their reuse
Tenc = 477                               # one 512-token chunk becomes 477 token rows
enc_rng     = stage_range(Tenc, BGE_M3)

decode_AI, prefill_AI, enc_AI = decode_rng[1], prefill_rng[0], enc_rng[0]
for stg, (lo, hi) in [('decode', decode_rng), ('encoder', enc_rng), ('prefill', prefill_rng)]:
    side = 'memory-bound' if hi < ridge else ('compute-bound' if lo > ridge else 'near the ridge')
    print(f'{stg:8s} AI ~ {lo:6.1f} to {hi:6.1f} FLOP/byte  -> {side}')

print(f'prefill K/V projections: AI ~ {kv_prefill:.1f} FLOP/byte (N = 256)')
print('\nDecode is two orders of magnitude below the ridge; the prefill Q/O and MLP layers are above it.')

# --- self-check: decode sits below the ridge, the prefill Q/O and MLP layers above it ---
prefill_qo_mlp = prefill_rng[0]
assert decode_AI < ridge < prefill_qo_mlp, \
    f'roofline ordering wrong: decode={decode_AI:.1f} ridge={ridge:.0f} prefill={prefill_qo_mlp:.1f}'
print(f'self-check passed: decode AI {decode_AI:.1f} < ridge {ridge:.0f} < prefill Q/O and MLP AI {prefill_qo_mlp:.1f}')


## 4. Mixed precision (FP16): prefill against decode

Principle #14 says Tensor Cores pay only where arithmetic is the bottleneck. We test that directly by
timing a **prefill-shaped** matmul (compute-bound) and a **decode-shaped** matvec (memory-bound) in
FP32 vs FP16.

- Prefill-shaped: a large matrix times a wide activation block (many tokens). FP16 should win clearly.
- Decode-shaped: the same weight matrix times a single-token vector. FP16 barely helps the math,
  because the kernel is waiting on weight reads; the only real gain there is reading half the bytes.

In [ ]:
H = 4096                 # hidden size proxy
FFN = 4 * H
Ttok = 512               # prefill block

def bench_linear(dtype, tokens):
    W = torch.randn(FFN, H, device=dev, dtype=dtype)
    Xb = torch.randn(H, tokens, device=dev, dtype=dtype)
    return cuda_time(lambda: torch.matmul(W, Xb))

# prefill-shaped (compute-bound)
p32 = bench_linear(torch.float32, Ttok)
p16 = bench_linear(torch.float16, Ttok)
print(f'prefill-shaped matmul  fp32 {p32:.3f} ms | fp16 {p16:.3f} ms | speedup {p32/p16:.2f}x')

# decode-shaped (memory-bound: 1 token)
d32 = bench_linear(torch.float32, 1)
d16 = bench_linear(torch.float16, 1)
print(f'decode-shaped  matvec  fp32 {d32:.3f} ms | fp16 {d16:.3f} ms | speedup {d32/d16:.2f}x')

print('\nExpect a solid speedup on the prefill-shaped matmul and a much smaller one on the',
      'decode-shaped matvec. The decode gain that does appear comes from moving fewer bytes',
      '(fp16 weights), not from faster tensor-core math, the memory-traffic lever of Chapter 9.')

# --- self-check: fp16 helps the compute-bound prefill more than the memory-bound decode ---
_prefill_speedup = p32 / p16
_decode_speedup = d32 / d16
assert _prefill_speedup > _decode_speedup, \
    f'expected prefill to gain more from fp16, got prefill={_prefill_speedup:.2f}x decode={_decode_speedup:.2f}x'
print(f'self-check passed: fp16 prefill speedup {_prefill_speedup:.2f}x > decode speedup {_decode_speedup:.2f}x')


### 4b. Optional: measure it on the real RAG models

This cell loads the actual case-study models and times prefill in FP32 vs FP16 and one decode step.
It downloads weights, so it is gated behind a flag. It reports the measured prefill change and shows
that decode is essentially unchanged, so we do **not** claim an end-to-end speedup from this lever.

In [ ]:
RUN_REAL_MODELS = False  # set True to download Gemma-2B-IT and time it

if RUN_REAL_MODELS:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    mid = 'google/gemma-2b-it'
    tok = AutoTokenizer.from_pretrained(mid)
    prompt = 'Summarize the role of memory bandwidth in transformer inference.' * 8
    ids = tok(prompt, return_tensors='pt').input_ids.to(dev)

    def time_prefill(dtype):
        m = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=dtype).to(dev).eval()
        with torch.no_grad():
            t = cuda_time(lambda: m(ids).logits, iters=10, warmup=3)
        del m; gc.collect(); torch.cuda.empty_cache()
        return t

    pf32 = time_prefill(torch.float32)
    pf16 = time_prefill(torch.float16)
    print(f'real prefill: fp32 {pf32:.1f} ms | fp16 {pf16:.1f} ms | speedup {pf32/pf16:.2f}x')
    print('Decode is dominated by per-token weight reads; halving weight bytes (fp16) helps via',
          'memory traffic, not tensor-core compute. See Chapter 9 for the accuracy-aware version.')
else:
    print('Set RUN_REAL_MODELS = True to run the full model measurement.')

## 5. Plot the roofline for your GPU

The cell below draws the roofline of the detected card with the three RAG stages placed by their arithmetic
intensity. The chapter's printed figures come from the case-study measurements in `case_study/reference_results/`.


In [ ]:
import matplotlib.pyplot as plt

bw = PEAK_GBs * 1e9; peak = PEAK_TFLOPS * 1e12
AI = np.logspace(-1, 3.2, 400)
perf = np.minimum(AI * bw, peak) / 1e12
plt.figure(figsize=(7, 4.2))
plt.plot(AI, perf, lw=2.4, color='#404040')
plt.axvline(ridge, ls='--', color='#999999')
for lab, ai in [('decode', decode_AI), ('encoder', enc_AI), ('prefill', prefill_AI)]:
    y = min(ai * bw, peak) / 1e12
    plt.scatter([ai], [y], s=90, zorder=5)
    plt.annotate(f'{lab}\nAI~{ai:.0f}', (ai, y), textcoords='offset points', xytext=(6, -22))
plt.xscale('log'); plt.yscale('log')
plt.xlabel('arithmetic intensity (FLOP/byte)'); plt.ylabel('TFLOP/s')
plt.title('Roofline with the RAG stages (your hardware)')
plt.tight_layout(); plt.show()

print(f'Roofline drawn for {key} datasheet peaks.')

## 6. Exercises

These build on the case study from earlier chapters. The sections named below are the sections of this notebook.

These five notebook exercises are separate from the chapter's Exercises 5.1 to 5.8, which are in the book's online
resources. Each notebook exercise is a short version of one of them:

| Notebook exercise | Related chapter exercise |
|---|---|
| 1: copy bandwidth against size | Exercise 5.3: Build empirical roofs |
| 2: stride against slowdown | Exercise 5.2: Correct coalescing |
| 3: arithmetic intensity of your own model | Exercise 5.4: Dense-linear AI versus M |
| 4: token count at the ridge point | Exercise 5.4: Dense-linear AI versus M |
| 5: FP32 against FP16 decode steps | Exercise 5.5: Mixed precision with evidence |

**Notebook exercise 1.** Change `N` in Section 1 across `2**20 ... 2**28` and plot effective copy bandwidth vs size.
At what size does bandwidth plateau, and what does that say about cache effects?

**Notebook exercise 2.** In Section 2, sweep `stride` over `1, 2, 8, 32, 128`. Plot slowdown vs stride and explain
the curve in terms of transaction width.

**Notebook exercise 3.** Using your own model's parameter count and a prefill length you choose, recompute the three
arithmetic intensities and decide, before timing, which stages FP16 can speed up. Then verify with
Section 4.

**Notebook exercise 4.** Vary `Ttok` in Section 4 from `1` to `1024`. Find the token count at which the prefill-shaped
matmul crosses from memory-bound to compute-bound on your card, and relate it to the ridge point.

**Notebook exercise 5.** (Stretch) Set `RUN_REAL_MODELS = True`, then add a cell that times 32 decode steps in FP32
and FP16. Confirm the per-step time is set by weight reads and report whether FP16 weights change it,
and by how much. Tie your result back to Principle #14.